# Ejercicio 2: tres soluciones sobre los mismos 20 viajes

El mismo `viajes_features.csv` y la misma preparación importada de `preparacion_taxis.py`. Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo tienen la comprobación. La última celda imprime una comprobación por paso; no la modifiques.

1. Completa los pasos 1 a 3 y corre hasta la celda que escribe `prompt_llm.txt` (30 viajes con su tarifa como ejemplos, y 20 sin tarifa).
2. Copia `prompt_llm.txt` completo en un LLM de chat (el que uses: ChatGPT, Claude, Gemini u otro). Guarda los 20 números que responda, uno por línea, en `respuestas_llm.txt`, en esta misma carpeta.
3. Completa los pasos 4 y 5 y córrelos. Los modelos ya están entrenados: no hace falta volver a correr las celdas de arriba.

In [1]:
import time

import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from preparacion_taxis import dividir, preparacion

viajes = pd.read_csv("viajes_features.csv")
viajes["hora"] = pd.to_datetime(viajes["fecha_hora_inicio"]).dt.hour
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes, columnas_extra=["hora"])
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (2380, 11) Prueba: (595, 11)


In [2]:
# Los mismos 20 viajes de prueba para las tres soluciones, y 30 viajes de
# entrenamiento como ejemplos para el LLM.
muestra = X_prueba.sample(20, random_state=7)
y_muestra = y_prueba.loc[muestra.index]
ejemplos = X_ent.sample(30, random_state=7)
y_ejemplos = y_ent.loc[ejemplos.index]

## La función que evalúa cada modelo

In [3]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    """Entrena el modelo despues de la preparacion y mide el error absoluto
    medio en entrenamiento y en prueba. Guarda la fila de la tabla en
    resultados y el pipeline entrenado en entrenados, los dos con su
    nombre: si vuelves a correr la celda de un modelo, se reemplazan, no
    quedan repetidos."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    resultados[nombre] = {
        "modelo": nombre,
        "mae_entrenamiento": round(mae_ent, 4),
        "mae_prueba": round(mae_prueba, 4),
        "brecha": round(mae_prueba - mae_ent, 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:28s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}   {segundos:6.2f} s")

## Paso 1: la red neuronal

In [4]:
# TODO 1: la red neuronal del Ejercicio 1, con el nombre "red neuronal".
# Deja el pipeline entrenado en red.
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42))
red = entrenados["red neuronal"]

red neuronal                 entrenamiento  1.0882   prueba  1.1695     4.44 s


## Paso 2: el ensamblaje

In [5]:
# TODO 2: el ensamblaje del Ejercicio 1, con el nombre "gradient boosting".
# Deja el pipeline entrenado en ensamblaje.
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))
ensamblaje = entrenados["gradient boosting"]

gradient boosting            entrenamiento  1.0705   prueba  1.2754     0.28 s


## Paso 3: los dos modelos sobre los 20 viajes

In [6]:
# TODO 3: mae_red_muestra y mae_ensamblaje_muestra, el error absoluto
# medio de cada modelo sobre los 20 viajes de muestra.
mae_red_muestra = mean_absolute_error(y_muestra, red.predict(muestra))
mae_ensamblaje_muestra = mean_absolute_error(y_muestra, ensamblaje.predict(muestra))
print(f"En los 20 viajes: red neuronal {mae_red_muestra:.4f}   gradient boosting {mae_ensamblaje_muestra:.4f}")

En los 20 viajes: red neuronal 0.9637   gradient boosting 0.8957


## El prompt para el LLM

In [7]:
LEGIBLES = ["zona_origen", "zona_destino", "distancia_km", "pasajeros", "hora", "es_fin_de_semana", "metodo_pago"]


def escribir_prompt():
    filas_ejemplo = ejemplos[LEGIBLES].copy()
    filas_ejemplo["tarifa"] = y_ejemplos.values
    texto = (
        "Estimas tarifas de taxi en dolares. Abajo hay 30 viajes con su tarifa real y "
        "20 viajes sin tarifa. Responde solo con 20 numeros, uno por linea, en el mismo "
        "orden de los viajes sin tarifa, sin texto adicional. La distancia esta en "
        "kilometros; si falta, el valor esta vacio.\n\n"
        "Viajes con tarifa:\n"
        + filas_ejemplo.to_csv(index=False)
        + "\nViajes sin tarifa:\n"
        + muestra[LEGIBLES].to_csv(index=False)
    )
    with open("prompt_llm.txt", "w") as archivo:
        archivo.write(texto)
    return texto


prompt = escribir_prompt()
print("prompt_llm.txt escrito:", len(prompt), "caracteres")

prompt_llm.txt escrito: 2902 caracteres


## Paso 4: las respuestas del LLM

In [8]:
# predicciones_llm, la lista de 20 numeros de respuestas_llm.txt, y
# mae_llm, su error absoluto medio sobre los mismos 20 viajes.
with open("respuestas_llm.txt") as archivo:
    predicciones_llm = [float(linea.strip().replace("$", "")) for linea in archivo if linea.strip()]
mae_llm = mean_absolute_error(y_muestra, predicciones_llm)
print(f"LLM en los 20 viajes: {mae_llm:.4f}")

LLM en los 20 viajes: 1.1375


## Paso 5: el costo del LLM

In [9]:
# El costo de estimar 10 000 viajes al mes con el LLM, en dolares, en
# costo_10000. Busca en la pagina de precios del proveedor que usaste el
# precio por millon de tokens de entrada y anotalo abajo. Cuenta un token
# cada 4 caracteres, y un prompt como este por cada 20 viajes; ignora los
# tokens de salida.
PRECIO_POR_MILLON_DE_TOKENS = 4.00  # Claude Opus 5.5, entrada, tarifa estandar de la API
tokens_por_prompt = len(prompt) / 4
prompts_al_mes = 10_000 / 20
costo_10000 = tokens_por_prompt * prompts_al_mes * PRECIO_POR_MILLON_DE_TOKENS / 1_000_000
print(f"Tokens por prompt: {tokens_por_prompt:.0f} | costo de 10 000 viajes al mes: {costo_10000:.4f} dolares")

Tokens por prompt: 726 | costo de 10 000 viajes al mes: 1.4510 dolares


## Comprobación

In [10]:
import os


def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


comprobar(1, red is not None and red.named_steps["modelo"].n_iter_ < 1000)
comprobar(2, ensamblaje is not None and isinstance(ensamblaje.named_steps["modelo"], GradientBoostingRegressor))
comprobar(3, mae_red_muestra is not None and mae_ensamblaje_muestra is not None
          and 0 < mae_red_muestra < 3 and 0 < mae_ensamblaje_muestra < 3)
comprobar(4, os.path.exists("respuestas_llm.txt") and predicciones_llm is not None
          and len(predicciones_llm) == 20 and mae_llm is not None)
comprobar(5, PRECIO_POR_MILLON_DE_TOKENS is not None and costo_10000 is not None and costo_10000 > 0)

Paso 1: OK
Paso 2: OK
Paso 3: OK
Paso 4: OK
Paso 5: OK
